# SPOTPY calibration steps

**Code under development**
Note: sampler needs set-up and debugging

## Setup

1. Load raven files to Data/Calibration/{model name}


In [ ]:

import os
import sys
import pandas as pd
import spotpy as sp

# Ensure the project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

In [ ]:
# Check for Raven.exe in the DATA_CAL folder
raven_exe_path = os.path.join(DATA_CAL, "Raven.exe")
raven_name = "CypressRiverT_ERA5"

## Raven test run

In [ ]:
# raven_name = input("Enter the Raven model name (if different):")
if os.path.isfile(raven_exe_path):
    test_confirm = input(f"Raven.exe found at {raven_exe_path}. Do you want to run it? (yes or y to confirm): ")
    if (test_confirm.lower() != 'yes' and test_confirm.lower() != 'y'):
        print("Exiting without running Raven.exe.")
    else:
        print(f"Testing Raven.exe at {raven_exe_path}...")
        # Set working directory to DATA_CAL and run Raven.exe
        rv.run_raven(raven_name, DATA_CAL)
else:
    print(f"Raven.exe not found in location: {DATA_CAL}")

## Calibration set-up and test run



In [ ]:
calibrator = spot_setup_rvn(raven_name, DATA_CAL, obj_fn= "nse", split_areas = "None", spring_eval= True, low_seep=True, routing = True, mpi_cores= 0)


In [ ]:
# Use the Pareto Archived Dynamically Dimensioned Search (PADDS) algorithm for calibration
# sampler=sp.algorithms.padds(spot_setup=calibrator, # Core model setup object
#                             dbname = "padds_raven", # Save results of cal run to this database
#                             dbformat='csv')

 
sampler=sp.algorithms.mcmc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
                           parallel='seq',
                           dbname='test',
                           dbformat='csv')

# sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
#                            parallel='seq',
#                            dbname='mc_raven',
#                            dbformat='csv')



In [ ]:
rep = 5
sampler.sample(rep)

## REMINDER - try to reduce the evaluation period to 2003 onward to improve performance

# setup = calibrator  # your spot_setup object

# params = [p.optguess for p in setup.parameters]
# print("params:", params)

# sim = setup.simulation(params)
# evals = setup.evaluation()
# obj = setup.objectivefunction(sim, evals)
# print("Objective output:", obj)

In [ ]:
# results_df = sp.analyser.load_csv_results(sampler.dbname)
results_df = sp.analyser.load_csv_results("test")

In [ ]:
param_nse_sets = results_df[results_df.columns[0:7]]
param_nse_sets.head()

## FAST Analysis

Run fast method in mpiexec

With 5 parameters, the number of reps required for FAST is 2245

In [ ]:
fast_results = sp.analyser.load_csv_results("../fast_routing20250815_11")

In [ ]:
sp.analyser.plot_fast_sensitivity(_data, number_of_sensitiv_pars=3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def sensitivity_plot(_data, x_str, y_str, objective_str, filename = None):
    # Extract the first 3 columns from _data
    x = _data[x_str]
    y = _data[y_str]
    objective = _data[objective_str]

    # Normalize objective for color mapping
    norm_objective = (objective - objective.min()) / (objective.max() - objective.min())

    # Create a color map: blue for low objective, red for high objective
    colors = plt.cm.coolwarm(norm_objective)

    plt.figure(figsize=(8,6))
    scatter = plt.scatter(x, y, c=colors, s=40, edgecolor='k')
    plt.xlabel(x_str)
    plt.ylabel(y_str)
    plt.title(f'FAST Results: {x_str} vs {y_str} colored by {objective_str}')

    # Add colorbar to show objective scale
    sm = plt.cm.ScalarMappable(cmap=plt.cm.coolwarm, norm=plt.Normalize(vmin=objective.min(), vmax=objective.max()))
    sm.set_array([])
    # cbar = plt.colorbar(sm)
    # cbar.set_label('objective')

    if filename:
        plt.savefig(filename)
        print(f"Plot saved as {filename}")

    plt.show()

# sensitivity_plot(fast_results, 'parseep_k', 'partc_tp_fact', 'like1')

In [ ]:
parameters = ['parseep_k', 'partc_tp_fact', 'part_peak_fact', 'parqref_scale', 'parbedslope_mult']

parameter_tuples = [(parameters[i], parameters[j]) for i in range(len(parameters)) for j in range(i + 1, len(parameters))]

print(parameter_tuples)

In [ ]:
for param1, param2 in parameter_tuples:
    sensitivity_plot(fast_results, param1, param2, 'like1', filename=f"fast_sensitivity_{param1}_{param2}.png") # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">

## Calibration Analysis

In [ ]:
calibration_results = sp.analyser.load_csv_results("../rope20250817_09")

calibration_prior = calibration_results[:1000]

calibration_post = calibration_results[9000:]

In [ ]:

calibration_post_best = pd.DataFrame(calibration_results[7500:]).sort_values(by='like1', ascending=False).head(500)

In [ ]:
parameters = ['pardep_fact', 'parseep_k', 'parbasin_fact', 'partc_tp_fact', 'part_peak_fact', 'parqref_scale', 'parbedslope_mult']

parameter_tuples = [(parameters[i], parameters[j]) for i in range(len(parameters)) for j in range(i + 1, len(parameters))]

print(parameter_tuples)

In [ ]:
calibration_post_best.columns

In [ ]:
calibration_post_best.rename(columns = {"like1" : "NSE"}, inplace=True)
for col in calibration_post_best.columns:
    plt.hist(calibration_post_best[col], bins=10, alpha=0.7, label=f'count {col}')
    plt.legend()
    plt.title(f'Distribution of {col} in top 500 of the final 2500 ROPE simulations')
    median_col = np.round(calibration_post_best[col].median(), 4)
    mean_col = np.round(calibration_post_best[col].mean(), 4)
    plt.xlabel(f'median: {median_col}, mean: {mean_col}')
    plt.show()

In [ ]:
for param1, param2 in parameter_tuples:
    sensitivity_plot(calibration_post_best, param1, param2, 'like1', filename=f"rope_distribution_{param1}_{param2}.png") # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">

In [ ]:
for param1, param2 in parameter_tuples:
        print(f'Parameter 1: {param1}, Parameter 2: {param2}, corr:{calibration_post_best[[param1, param2]].corr().iloc[0, 1]}')

## Additional workflows

- Extract Raven results

- Specify calibration parameters, method, iterations, etc.
- Run calibration
- Store results & optimal parameters

## Update Raven files

Testing channel routing - Full parameter reset

In [ ]:
Cypress_rvp = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvp")
Cypress_rvp.rvp_update("no_split", True, \
                        pd.Series({"seep_k": 0.01, "dep_fact" : 1, "bedslope_mult": 1}))

Cypress_channels_rvp = rv.RavenFile(
    "channel_properties_T_ERA5", calib_dir= DATA_CAL, extension = "rvp")
Cypress_channels_rvp.rvp_update("no_split", False, \
                        pd.Series({"seep_k": 0.01, "dep_fact" : 1, "bedslope_mult": 1}))

Cypress_rvh = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvh")

Cypress_rvh.rvh_update("no_split", True, True, pd.Series({"seep_k": 0.01, "basin_fact": 1, "t_peak_fact": 1, "tc_tp_fact": 1, "bedslope_mult": 1, "qref_scale": 1}))

In [ ]:
rv.run_raven(raven_name, DATA_CAL)

Testing channel routing - exaggerated parameter values

In [ ]:
Cypress_rvp.rvp_update("no_split", False, \
                        pd.Series({"seep_k": 0.01, "dep_fact" : 1, "bedslope_mult": 1}))

Cypress_channels_rvp.rvp_update("no_split", False, \
                        pd.Series({"seep_k": 0.01, "dep_fact" : 1, "bedslope_mult": 1}))

Cypress_rvh.rvh_update("no_split", False, True, pd.Series({"seep_k": 0.01, "basin_fact": 1, "t_peak_fact": 1, "tc_tp_fact": 1, "bedslope_mult": 1, "qref_scale": 0.2}))

In [ ]:
rv.run_raven(raven_name, DATA_CAL)

Manual Raven results with same parameters yield a different NSE

### Simulation with median ROPE results

In [ ]:
# param_vect = pd.Series({'dep_fact':0.8, 'seep_k':0.025, 'basin_fact':0.82, 't_peak_fact':0.26, 'tc_tp_fact':2.1, 'bedslope_mult':4.2, 'qref_scale':0.73}) # Median of each value from ROPE best post

param_vect = pd.Series({'dep_fact':0.79, 'seep_k':0.002, 'basin_fact':0.89, 't_peak_fact':0.27, 'tc_tp_fact':2.4, 'bedslope_mult':4, 'qref_scale':0.45}) # High performing ROPE sim with high basin scaling

In [ ]:
Cypress_rvp = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvp")
Cypress_rvp.rvp_update("no_split", True, \
                        param_vect)

Cypress_channels_rvp = rv.RavenFile(
    "channel_properties_T_ERA5", calib_dir= DATA_CAL, extension = "rvp")
Cypress_channels_rvp.rvp_update("no_split", True, \
                        param_vect)

Cypress_rvh = rv.RavenFile(
    raven_name, calib_dir= DATA_CAL, extension = "rvh")

Cypress_rvh.rvh_update("no_split", True, True, param_vect)

In [ ]:
rv.run_raven(raven_name, DATA_CAL)

ROPE median best simulation yielded a spring NSE of 0.389, but -39% bias
ROPE best simulation with high basin scaling (0.89) yielded  NSE 0.37 with -13% bias

### Hydrographs

In [ ]:
test = rv.RavenHydrograph(DATA_CAL)
sim = test.read_sim(subbasin_id = "sub875", start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True)
eval = test.read_obs(start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True)

In [ ]:
def read_obs(self, start_yr, end_yr, spring_only = True, refresh=False) -> pd.DataFrame:
    """
    Extract observed discharge data from the hydrograph DataFrame.

    Returns:
    ----------
    observed (pd.Series): Time series of observed discharge values.
    """
    if refresh:
        self.data = self.read_hydrograph()
        # self.data = self.data.loc[self.data["date"].dt.year.between(start_yr, end_yr)]

    self.eval = self.data.loc[self.data["date_time"].dt.year.between(start_yr, end_yr)]

    if spring_only:
        self.eval = self.eval.loc[self.eval["date_time"].dt.month.lt(7)]
    
    noon_index = self.eval.index[self.eval["date_time"].dt.hour == 12]

    noon_index_interval = noon_index[1]-noon_index[0]

    # Look for any column with 'observed' in its name
    for col in self.eval.columns:
        if "observed" in col.lower():
            # removed values in this column that are not at noon
            observed = self.eval[col]
            observed = observed.where(observed.index.isin(noon_index))
            observed = observed.interpolate(method='linear', limit_direction='both', limit = noon_index_interval, limit_area = "inside")
            
            break
    return observed

In [ ]:
eval_test = test.read_obs(start_yr=2003, end_yr=2021, spring_only = False, refresh=True).reset_index(drop=True)

In [ ]:
non_null = eval.notnull()
        
valid_sim = sim[non_null]
valid_eval = eval[non_null]


In [ ]:
valid_start = eval[eval.notnull()].index[0]

print(valid_start)

values_start = valid_eval[valid_eval != 0].index[0]

eval[values_start:(values_start + 24)]





In [ ]:
test.data.iloc[1775]

In [ ]:

nse = 1- np.sum((valid_sim - valid_eval) ** 2) / np.sum((valid_eval - np.mean(valid_eval)) ** 2)
nse = np.squeeze(nse).item()
print("Nash-Sutcliffe Efficiency (NSE):", nse)